In [ ]:
from oqd_compiler_infrastructure import CFGBlockAccumulator, Post, PrettyPrint

from oqd_core.frontend.atomic import parse_atomic

printer = Post(PrettyPrint())

with open("test.atomic", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_atomic(source)
print(printer(circuit))

AtomicCircuit
  - statements: list
    - 0: Declaration
      - name: str(ions)
      - value: IonRegister
        - size: Constant
          - value: int(2)
    - 1: Declaration
      - name: str(q0)
      - value: Extract
        - value: Access
          - name: str(ions)
        - index: Constant
          - value: int(0)
    - 2: Declaration
      - name: str(q1)
      - value: Extract
        - value: Access
          - name: str(ions)
        - index: Constant
          - value: int(1)
    - 3: Declaration
      - name: str(targets)
      - value: AtomicList
        - values: list
          - 0: Access
            - name: str(q0)
          - 1: Access
            - name: str(q1)
    - 4: Declaration
      - name: str(pi)
      - value: Constant
        - value: float(3.14159)
    - 5: Declaration
      - name: str(omega)
      - value: RuntimeVar
        - name: str(#omega)
    - 6: Declaration
      - name: str(phi)
      - value: RuntimeVar
        - name: str(#phi)
    - 7: D

In [2]:
from oqd_compiler_infrastructure import CFGBlockAccumulator, Post, PrettyPrint

from oqd_core.analysis.atomic.cfg import AtomicCFGBuilder
from oqd_core.analysis.atomic.symbol_table import AtomicSymbolTableBuilder
from oqd_core.analysis.atomic.type_checker import AtomicTypeChecker
from oqd_core.frontend.atomic import parse_atomic

printer = Post(PrettyPrint())

with open("test.atomic", mode="r", encoding="utf8") as f:
    source = f.read()

circuit = parse_atomic(source)
cfg = AtomicCFGBuilder()(circuit)
cfg = CFGBlockAccumulator()(cfg)
checker = AtomicTypeChecker(cfg)
symbol_table = AtomicSymbolTableBuilder(cfg, checker.dataflow_result).symbol_table

ImportError: cannot import name 'BoolAnd' from 'oqd_core.interface.atomic' (/Users/vishnu/Documents/OQD/oqd-core/src/oqd_core/interface/atomic/__init__.py)

In [ ]:
from oqd_core.compiler.atomic.passes.compile import compile_atomic_circuit

circuit, cfg = compile_atomic_circuit(
    circuit, cfg, symbol_table, checker.dataflow_result
)

In [ ]:
circuit

AtomicCircuit(class_='AtomicCircuit', statements=[Declaration(class_='Declaration', name='ions', value=IonRegister(class_='IonRegister', size=2)), Declaration(class_='Declaration', name='q0', value=Extract(class_='Extract', access=Access(class_='Access', name='ions'), index=0)), Declaration(class_='Declaration', name='q1', value=Extract(class_='Extract', access=Access(class_='Access', name='ions'), index=1)), Declaration(class_='Declaration', name='targets', value=AtomicList(class_='AtomicList', values=[Access(class_='Access', name='q0'), Access(class_='Access', name='q1')])), Declaration(class_='Declaration', name='pi', value=MathNum(class_='MathNum', value=3.14159)), Declaration(class_='Declaration', name='omega', value=MathVar(class_='MathVar', name='#omega')), Declaration(class_='Declaration', name='phi', value=MathVar(class_='MathVar', name='#phi')), Declaration(class_='Declaration', name='t_gate', value=MathNum(class_='MathNum', value=1e-05)), Declaration(class_='Declaration', na

In [ ]:
from oqd_compiler_infrastructure import cfg_to_dot

dot = cfg_to_dot(cfg, serialize=serialize_atomic, max_lines=5)
print(dot.source)
dot.render("cfg", format="png", cleanup=True)

digraph {
	0 [label=" Block #0
------------------------
"]
	0 -> 1
	1 [label=" Block #1
------------------------
ions = ionreg(2)
q0 = ions[0]
q1 = ions[1]
targets = [q0, q1]
pi = 3.14159
..."]
	1 -> 38
	38 [label="Branch Block #38
------------------------
Condition: flag"]
	38 -> 39
	38 -> 41
	39 [label=" Block #39
------------------------
serial {
pulse(beam_mw, 2e-6, targets, false)
}
pulse(beam_mw, 2e-6, targets, false)"]
	39 -> 43
	41 [label=" Block #41
------------------------
serial {
pulse(beam_det, 2e-5, targets, true)
}
pulse(beam_det, 2e-5, targets, true)"]
	41 -> 43
	43 [label="Branch Block #43
------------------------
Condition: n > 0"]
	43 -> 44
	43 -> 52
	44 [label=" Block #44
------------------------
serial {
parallel {
pulse(beam_mw, 1e-6, targets, false)
}
}
parallel {
pulse(beam_mw, 1e-6, targets, false)
}
pulse(beam_mw, 1e-6, targets, false)
n = -1 + n"]
	44 -> 48
	48 [label="Branch Block #48
------------------------
Condition: n == 1"]
	48 -> 49
	48 -> 50
	49 [labe

'cfg.png'